# 🏢 Information Extraction from Polish Business Documents
### Interactive Demo: Broker vs. Client Disambiguation & Evaluation

This notebook demonstrates the end-to-end extraction of structured contract parties (**Broker** vs. **Client**) and their respective **NIP** numbers from Polish business documents (e.g. *Pełnomocnictwa brokerskie*, insurance agreements).

#### Pipeline Architecture:
1. **Neural Zero-Shot Entity Extraction (`GLiNER`)**: Detects base entity mentions (`company`, `NIP`).
2. **Neuro-Symbolic Post-Processing (`EntityPostprocessor`)**: Strips colon/label prefixes (`"NIP: 521..."` $\to$ `"521..."`) and enforces digit length rules.
3. **Syntactic Binding & Mutual Exclusivity Resolution (`BrokerClientResolver`)**: Binds NIPs syntactically to companies and resolves asymmetric legal roles (`broker` vs `client`).


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Ensure project src/ is on PYTHONPATH
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from structured_ie.evaluation.evaluator import PipelineEvaluator
from structured_ie.pipeline.broker_client_pipeline import BrokerClientPipeline
from structured_ie.postprocessing.entity_cleaner import EntityPostprocessor

pd.set_option("display.max_colwidth", None)
print("Libraries imported successfully!")

/home/mrzeleq/Projekty/structured-information-extraction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


/home/mrzeleq/Projekty/structured-information-extraction/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 1. 🚀 Initialize the Extraction Pipeline
We instantiate `BrokerClientPipeline`, which loads the multilingual zero-shot GLiNER model (`urchade/gliner_multi-v2.1`) and pairs it with the neuro-symbolic resolver.


In [2]:
print("Loading BrokerClientPipeline with urchade/gliner_multi-v2.1...")
pipeline = BrokerClientPipeline.from_pretrained(gliner_model_name="urchade/gliner_multi-v2.1")
print("✅ BrokerClientPipeline ready for inference!")

/home/mrzeleq/Projekty/structured-information-extraction/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Loading BrokerClientPipeline with urchade/gliner_multi-v2.1...


Fetching 5 files:   0%|                                  | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|████████████████████████| 5/5 [00:00<00:00, 1061.04it/s]

✅ BrokerClientPipeline ready for inference!


## 2. 📄 Interactive Example 1: Standard Broker Authorization (*Pełnomocnictwo Brokerskie*)
A typical scenario where a client company grants insurance brokerage power of attorney to a broker.


In [3]:
sample_text_1 = (
    "Mocodawca: Alfa Logistyka Sp. z o.o. (NIP: 5213456789) udziela"
    " pełnomocnictwa firmie Mentor Broker Sp. z o.o. (NIP: 5252525252) do"
    " reprezentowania przed zakładami ubezpieczeń i negocjowania warunków"
    " umów ubezpieczenia majątkowego."
)

print("--- Input Document ---")
print(sample_text_1)

# Extract high-level business DTO
result_1 = pipeline.extract_structured(sample_text_1)

--- Input Document ---
Mocodawca: Alfa Logistyka Sp. z o.o. (NIP: 5213456789) udziela pełnomocnictwa firmie Mentor Broker Sp. z o.o. (NIP: 5252525252) do reprezentowania przed zakładami ubezpieczeń i negocjowania warunków umów ubezpieczenia majątkowego.


In [4]:
# Visualize extracted parties in a structured summary table
parties_df = pd.DataFrame(
    [
        {
            "Role": "🛡️ Broker (Pośrednik)",
            "Company Name": result_1.broker.name if result_1.broker else "N/A",
            "NIP": result_1.broker.nip if result_1.broker else "N/A",
            "Confidence": f"{result_1.broker.score:.4f}" if result_1.broker else "N/A",
        },
        {
            "Role": "🏢 Client (Mocodawca / Klient)",
            "Company Name": result_1.client.name if result_1.client else "N/A",
            "NIP": result_1.client.nip if result_1.client else "N/A",
            "Confidence": f"{result_1.client.score:.4f}" if result_1.client else "N/A",
        },
    ]
)

display(parties_df)
print(f"\nOverall Extraction Confidence: {result_1.confidence_score:.4f}")
print("\nStructured JSON Output:")
print(json.dumps(result_1.to_dict(), indent=2, ensure_ascii=False))

,Role,Company Name,NIP,Confidence
0,🛡️ Broker (Pośrednik),Mentor Broker Sp. z o.o.,5252525252,0.9305
1,🏢 Client (Mocodawca / Klient),Alfa Logistyka Sp. z o.o.,5213456789,0.9291



Overall Extraction Confidence: 1.0000

Structured JSON Output:
{
  "broker": {
    "name": "Mentor Broker Sp. z o.o.",
    "nip": "5252525252",
    "name_span": [
      85,
      109
    ],
    "nip_span": [
      116,
      126
    ],
    "score": 0.9304892420768738
  },
  "client": {
    "name": "Alfa Logistyka Sp. z o.o.",
    "nip": "5213456789",
    "name_span": [
      11,
      36
    ],
    "nip_span": [
      43,
      53
    ],
    "score": 0.9291180372238159
  },
  "confidence": 1.0
}


## 3. 🔍 Interactive Example 2: Inverted Header / Key-Value Structure
Documents often feature tabular or key-value headers where the broker is declared first, followed by the client as *Ubezpieczający*.


In [5]:
sample_text_2 = (
    "Zaświadczenie o zawarciu polisy.\n"
    "Broker ubezpieczeniowy: Willis Polska Sp. z o.o., NIP 1112233445.\n"
    "Ubezpieczający (Klient): Budimex Konstrukcje S.A., NIP 9988776655.\n"
    "Zakres ochrony obejmuje ubezpieczenie OC działalności gospodarczej."
)

print("--- Input Document ---")
print(sample_text_2)

result_2 = pipeline.extract_structured(sample_text_2)

parties_df_2 = pd.DataFrame(
    [
        {
            "Role": "🛡️ Broker",
            "Company Name": result_2.broker.name if result_2.broker else "N/A",
            "NIP": result_2.broker.nip if result_2.broker else "N/A",
            "Confidence": f"{result_2.broker.score:.4f}" if result_2.broker else "N/A",
        },
        {
            "Role": "🏢 Client (Ubezpieczający)",
            "Company Name": result_2.client.name if result_2.client else "N/A",
            "NIP": result_2.client.nip if result_2.client else "N/A",
            "Confidence": f"{result_2.client.score:.4f}" if result_2.client else "N/A",
        },
    ]
)

display(parties_df_2)

--- Input Document ---
Zaświadczenie o zawarciu polisy.
Broker ubezpieczeniowy: Willis Polska Sp. z o.o., NIP 1112233445.
Ubezpieczający (Klient): Budimex Konstrukcje S.A., NIP 9988776655.
Zakres ochrony obejmuje ubezpieczenie OC działalności gospodarczej.


,Role,Company Name,NIP,Confidence
0,🛡️ Broker,Willis Polska Sp. z o.o.,1112233445,0.9193
1,🏢 Client (Ubezpieczający),Budimex Konstrukcje S.A.,9988776655,0.9292


## 4. 🔬 Ablation: Why Neuro-Symbolic Post-Processing is Crucial
Raw zero-shot NER models frequently capture prefix label tokens (`"NIP: 5213456789"` instead of `"5213456789"`) or mistake common nouns for entities.
Here we compare the raw GLiNER predictions directly against the post-processed output.


In [6]:
cleaner = EntityPostprocessor()
raw_entities = pipeline.ner_extractor.extract_entities(
    sample_text_1, ["company", "NIP"], threshold=0.3
)
cleaned_entities = cleaner.postprocess(raw_entities, sample_text_1)

ablation_data = []
for raw, clean in zip(raw_entities, cleaned_entities, strict=False):
    ablation_data.append(
        {
            "Raw Span (GLiNER)": f"'{raw.text}'",
            "Raw Label": raw.label,
            "Cleaned Span": f"'{clean.text}'",
            "Cleaned Label": clean.label,
            "Prefix / Noise Trimmed": "✂️ Yes" if raw.text != clean.text else "No (Clean)",
        }
    )

display(pd.DataFrame(ablation_data))

,Raw Span (GLiNER),Raw Label,Cleaned Span,Cleaned Label,Prefix / Noise Trimmed
0,'Alfa Logistyka Sp. z o.o.',company,'Alfa Logistyka Sp. z o.o.',company,No (Clean)
1,'NIP: 5213456789',NIP,'5213456789',NIP,✂️ Yes
2,'Mentor Broker Sp. z o.o.',company,'Mentor Broker Sp. z o.o.',company,No (Clean)
3,'NIP: 5252525252',NIP,'5252525252',NIP,✂️ Yes


## 5. 📊 In-Notebook Benchmark Evaluation (20 Real Polish Test Cases)
Now, let's run our test evaluation on the curated benchmark dataset `data/broker_client_cases.json`.
We evaluate in **Exact Match** mode (exact character boundaries and labels).


In [7]:
from structured_ie.evaluation.dataset import load_dataset_from_json

dataset_path = ROOT_DIR / "data" / "broker_client_cases.json"
print(f"Loading benchmark test cases from: {dataset_path.name}...")

samples = load_dataset_from_json(dataset_path)
print(f"Loaded {len(samples)} evaluation scenarios.")

entity_labels = ["broker", "client", "NIP"]
relation_labels = ["broker for", "has NIP"]

evaluator = PipelineEvaluator(
    entity_labels=entity_labels, relation_labels=relation_labels, match_mode="exact"
)

report = evaluator.evaluate(pipeline=pipeline, samples=samples)

print("✅ Benchmark Evaluation Completed!")
print("\n" + report.format_summary())

Loading benchmark test cases from: broker_client_cases.json...
Loaded 20 evaluation scenarios.


✅ Benchmark Evaluation Completed!

IE EVALUATION REPORT (Total Samples: 20)
Task                | Precision |  Recall   |    F1     | TP / FP / FN
------------------------------------------------------------
Entities (NER)      |  100.00%   |  100.00%   |  100.00%   | 80 /  0 /  0
Relations (RE)      |  100.00%   |  100.00%   |  100.00%   | 60 /  0 /  0


In [8]:
# Summary DataFrame
summary_df = pd.DataFrame(
    [
        {
            "Task": "Entities (NER)",
            "Precision": f"{report.entity_micro.precision:6.2%}",
            "Recall": f"{report.entity_micro.recall:6.2%}",
            "F1 Score": f"{report.entity_micro.f1:6.2%}",
            "TP": report.entity_micro.tp,
            "FP": report.entity_micro.fp,
            "FN": report.entity_micro.fn,
        },
        {
            "Task": "Relations (RE)",
            "Precision": f"{report.relation_micro.precision:6.2%}",
            "Recall": f"{report.relation_micro.recall:6.2%}",
            "F1 Score": f"{report.relation_micro.f1:6.2%}",
            "TP": report.relation_micro.tp,
            "FP": report.relation_micro.fp,
            "FN": report.relation_micro.fn,
        },
    ]
)

display(summary_df)

,Task,Precision,Recall,F1 Score,TP,FP,FN
0,Entities (NER),100.00%,100.00%,100.00%,80,0,0
1,Relations (RE),100.00%,100.00%,100.00%,60,0,0


## 6. 🔍 Case-by-Case Predictions vs. Ground Truth Inspection
Let's inspect the individual predictions across all 20 test cases to verify predictions against expectations.


In [9]:
inspection_rows = []
for idx, s in enumerate(samples, 1):
    text = s.text
    pred = pipeline.extract_structured(text)

    # Extract gold broker and client from case entities
    gold_broker = next((e.text for e in s.expected_entities if e.label == "broker"), "N/A")
    gold_client = next((e.text for e in s.expected_entities if e.label == "client"), "N/A")

    pred_broker = pred.broker.name if pred.broker else "N/A"
    pred_client = pred.client.name if pred.client else "N/A"

    broker_match = "✅ Match" if pred_broker == gold_broker else "❌ Mismatch"
    client_match = "✅ Match" if pred_client == gold_client else "❌ Mismatch"

    inspection_rows.append(
        {
            "Case": f"#{idx}",
            "Text Preview": text[:55] + "..." if len(text) > 55 else text,
            "Expected Broker": gold_broker,
            "Predicted Broker": pred_broker,
            "Broker Status": broker_match,
            "Expected Client": gold_client,
            "Predicted Client": pred_client,
            "Client Status": client_match,
        }
    )

df_cases = pd.DataFrame(inspection_rows)
display(df_cases)

,Case,Text Preview,Expected Broker,Predicted Broker,Broker Status,Expected Client,Predicted Client,Client Status
0,#1,Kancelaria Brokerska Nord Partner Sp. z o.o. (NIP: 5841...,Nord Partner Sp. z o.o.,Nord Partner Sp. z o.o.,✅ Match,Alfa Logistyka S.A.,Alfa Logistyka S.A.,✅ Match
1,#2,Mocodawca Krak-Bud Sp. z o.o. o numerze NIP 6760011223 ...,Vistula Insurance Broker Sp. k.,Vistula Insurance Broker Sp. k.,✅ Match,Krak-Bud Sp. z o.o.,Krak-Bud Sp. z o.o.,✅ Match
2,#3,Zleceniodawca Apex Trade Sp. z o.o. (NIP: 8990011223) u...,Omega Broker S.A.,Omega Broker S.A.,✅ Match,Apex Trade Sp. z o.o.,Apex Trade Sp. z o.o.,✅ Match
3,#4,Polisa wystawiona dla Ubezpieczającego: Budimex-Trans S...,Mentor S.A.,Mentor S.A.,✅ Match,Budimex-Trans Sp. z o.o.,Budimex-Trans Sp. z o.o.,✅ Match
4,#5,"Ja, prezes zarządu TechSoft Sp. z o.o. (NIP: 1112223344...",Grupa Brokerska Certus Sp. z o.o.,Grupa Brokerska Certus Sp. z o.o.,✅ Match,TechSoft Sp. z o.o.,TechSoft Sp. z o.o.,✅ Match
5,#6,Willis Towers Watson Polska Sp. z o.o. (NIP: 5260211789...,Willis Towers Watson Polska Sp. z o.o.,Willis Towers Watson Polska Sp. z o.o.,✅ Match,Fabryka Mebli Forte S.A.,Fabryka Mebli Forte S.A.,✅ Match
6,#7,Działając w imieniu poszkodowanego klienta Eko-Dachy Sp...,Dom Brokerski Aspergo S.A.,Dom Brokerski Aspergo S.A.,✅ Match,Eko-Dachy Sp. z o.o.,Eko-Dachy Sp. z o.o.,✅ Match
7,#8,"Informujemy, że jedynym wyłącznym brokerem obsługującym...",Kancelaria Brokerska Quantum Sp. z o.o.,Kancelaria Brokerska Quantum Sp. z o.o.,✅ Match,Solaris Logistics Sp. z o.o.,Solaris Logistics Sp. z o.o.,✅ Match
8,#9,Aneks nr 2 do umowy zawartej pomiędzy Klientem Euro-Tra...,Broker Finanse Consulting S.A.,Broker Finanse Consulting S.A.,✅ Match,Euro-Trans Cargo Sp. z o.o.,Euro-Trans Cargo Sp. z o.o.,✅ Match
9,#10,W imieniu Mocodawcy Selena Oil Sp. z o.o. NIP 897001234...,Maxima Sp. z o.o.,Maxima Sp. z o.o.,✅ Match,Selena Oil Sp. z o.o.,Selena Oil Sp. z o.o.,✅ Match
